# GalvCalc Demo

A walkthrough of the **GalvCalc** framework for modeling micro-galvanic
corrosion of alloys with coupled anodic dissolution and hydrogen evolution
kinetics.

This notebook demonstrates the five main modules:

1. **core** - bulk/surface structures and Nernst equilibrium potentials
2. **cathode** - surface properties, hydrogen adsorption, exchange-current (i_c0) estimation
3. **anode** - second-layer substitutional doping of anode surfaces
4. **polarization** - Butler-Volmer polarization curves, area-ratio analysis and alloy-content scans
5. **predictor** (optional) - CGCNN / TabPFN machine-learning interfaces

The companion notebook `test_10.4.ipynb` in this folder contains the extended
validation scripts used during development.

---

## 1. Core: bulk structures and equilibrium potentials

In [ ]:
# Silence noisy spglib deprecation warnings in the demo output.
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning, module="spglib")

# Make GalvCalc importable when running directly from the repository checkout.
# (When installed with `pip install GalvCalc`, this block is a no-op.)
try:
    import GalvCalc  # noqa: F401
except ImportError:
    import sys
    from pathlib import Path as _Path

    _root = None
    for _candidate in [_Path.cwd(), *_Path.cwd().parents]:
        if (_candidate / "GalvCalc" / "__init__.py").exists():
            _root = _candidate
            break
    if _root is None:
        raise ImportError(
            "GalvCalc is not installed and could not be located next to this notebook."
        )
    sys.path.insert(0, str(_root))
    import GalvCalc  # noqa: F401

import os
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from IPython.display import Image, display

# Headless-safe matplotlib backend (comment out in interactive sessions)
try:
    matplotlib.use("Agg")
except Exception:
    pass

# Directory for generated figures
Path("demo_output").mkdir(exist_ok=True)

# Keep the demo output clean: suppress third-party deprecation warnings and
# tqdm progress bars (remove in interactive use if you want to see them).
os.environ.setdefault("TQDM_DISABLE", "1")
warnings.filterwarnings("ignore")

from GalvCalc.core.structures import Bulk, Surface  # noqa: E402


In [ ]:
# ---- load a bulk structure (POSCAR / CIF / .vasp) ----
bulk = Bulk.from_file("POSCAR")
props = bulk.basic_properties()
print("Bulk properties:", props)
print("Lattice:", bulk.lattice)


In [ ]:
# ---- equilibrium potentials ----
# Single-ion reaction: Mg -> Mg[2+] + 2e-
Ee = Bulk.get_equilibrium_potential(
    ions="Mg[2+]",
    ion_numbers=[1],
    energy_formation=0.0,
)
print(f"Mg/Mg2+ E_eq = {Ee:.3f} V vs. SHE")

# Multi-ion compound dissolution: MgZn2 -> Mg[2+] + 2 Zn[2+] + 6e-
Ee2 = Bulk.get_equilibrium_potential(
    ions=["Mg[2+]", "Zn[2+]"],
    ion_numbers=[1, 2],
    energy_formation=-0.24,
)
print(f"MgZn2 E_eq = {Ee2:.3f} V vs. SHE")


### Surface generation

`Surface.from_bulk` builds a pymatgen slab from a bulk structure. The slab is
fully interoperable with the pymatgen ecosystem (`Slab`, `SlabGenerator`,
`AdsorbateSiteFinder`, ...).

In [ ]:
# Create the Mg2Si (111) slab used throughout the cathode examples
surface = Surface.from_bulk(
    bulk_structure=bulk,
    miller_index=(1, 1, 1),
    min_slab_size=10.0,
    min_vacuum_size=15.0,
    center_slab=True,
)
print(f"Surface area: {surface.surface_area:.2f} A^2")
print(f"Miller index: {surface.miller_index}")
print(f"Composition:  {surface.composition.reduced_formula}")
print(f"Atoms:        {len(surface)}")


### Adsorption-site schematic

Locating the candidate adsorption sites is the first step of the cathodic
hydrogen-evolution workflow. The figure below marks every adsorption site on
the (111) slab (marker **x** on the projected unit cell).

In [ ]:
# 1) locate all candidate adsorption sites with pymatgen's AdsorbateSiteFinder
ads_sites = surface.get_adsorption_sites()
all_sites = ads_sites.get("all", [])
print(f"Found {len(all_sites)} adsorption sites")
print("Site types:", {k: len(v) for k, v in ads_sites.items() if k != "all"})

# 2) schematic: slab + adsorption sites marked on the (111) surface
fig, ax = surface.visualize_adsorption_sites(
    adsorption_sites=ads_sites,
    marker_size=120,
    draw_unit_cell=True,
    save_path="demo_output/adsorption_sites_111.png",
)
plt.savefig("demo_output/adsorption_sites_111.png", dpi=150, bbox_inches="tight")
print("Saved demo_output/adsorption_sites_111.png")
display(Image(filename="demo_output/adsorption_sites_111.png"))


In [ ]:
# Generate the H-adsorbed structures for the first two sites
adsorbed = surface.generate_adsorption_structures("H", site_indices=[1, 2])
print(f"Generated {len(adsorbed)} adsorbed structures")
for idx, struct in adsorbed.items():
    print(f"  site {idx}: {struct.composition.formula}, {len(struct)} atoms")

# Each structure can be written to VASP POSCAR for DFT follow-up
# adsorbed[1].to(fmt="poscar", filename="H_on_Mg2Si_111_site1.vasp")


### Interactive 3D view (optional)

`Surface.visualize_3d()` opens an interactive VTK viewer. It requires the
optional `vtk` package and a GUI session, so it is skipped gracefully when
unavailable (e.g. in headless notebooks).

In [ ]:
# The interactive VTK viewer opens a native window and blocks until it
# is closed, so it is only invoked in interactive (non-Agg) sessions.
if matplotlib.get_backend().lower() == "agg":
    print("Headless backend detected; skipping the interactive 3D viewer.")
    print("In a GUI session, run:  surface.visualize_3d(show_unit_cell=True)")
else:
    try:
        surface.visualize_3d(show_unit_cell=True)
    except Exception as exc:
        print("3D viewer error:", exc)


---

## 2. Cathode: surface properties and hydrogen adsorption

`SurfaceProperties` batches the slab generation over all low-index Miller
indices and computes facet-resolved surface energies and work functions.

In [ ]:
from GalvCalc.cathode.surfaces import SurfaceProperties

# Generate all surfaces up to max_index = 1
surfaces = SurfaceProperties.from_bulk_structure(
    bulk,
    max_index=1,
    min_slab_size=10.0,
    min_vacuum_size=15.0,
)
print(f"Generated {len(surfaces.surfaces)} surface terminations")
for s in surfaces.surfaces:
    print(f"  - {s.full_name}  ({s.miller_index})")


In [ ]:
# Insert slab energies and work functions (normally from DFT) and build
# the facet-resolved properties DataFrame. The work functions are the DFT
# values of the manuscript's Mg2Si dataset; the slab/bulk energies are
# illustrative placeholders (replace them with your own VASP values).
from sympy import Symbol

# Bulk cell Mg8Si4 (4 f.u., total -33.6 eV -> -8.4 eV per f.u.); the (110)
# termination is slightly more stable than (111), as expected for the
# more closely packed surface.
slab_energies = {"Mg2Si_111_1": -23.6, "Mg2Si_110_1": -64.8}
work_funcs = {"Mg2Si_111_1": 2.9941, "Mg2Si_110_1": 3.7378}

df = surfaces.get_properties_dataframe(
    slab_energies=slab_energies,
    bulk_energy=-33.6,
    work_functions=work_funcs,
    delu_dict={Symbol("delu_Mg"): 0.0},
)
df[["full_name", "miller_index", "surface_energy_J_m2", "work_function_eV"]]


In [ ]:
# Most stable surface terminations (lowest surface energy)
stable = surfaces.get_most_stable_surfaces(n=3)
print("Most stable surfaces:")
for s in stable:
    print(f"  - {s.full_name}, surface energy = {s.surface_energy:.4f} J/m^2")


### Wulff construction

The Wulff shape of the crystal is built from the facet surface energies; the
resulting facet area fractions are the weights used for facet-averaged
properties (work function, exchange current density).

In [ ]:
# Construct the Wulff shape and save the plot + CSV summary
wulff = surfaces.wulff_construct(
    surface_energies=surfaces.surface_energies_dict,
    work_functions=surfaces.work_functions_dict,
    output_dir="demo_output",
    save_plot=True,
    save_csv=True,
    show_plot=False,
)
print("Facet area fractions:")
for hkl, area in wulff["area_fractions"].items():
    total = sum(wulff["area_fractions"].values())
    print(f"  {hkl}: {area / total * 100:.1f}%")
print("Saved demo_output/wulff_Mg2Si.png")
display(Image(filename="demo_output/wulff_Mg2Si.png"))


### Hydrogen adsorption analysis

`AdsorptionManager.H_adsorption_analysis` runs the complete workflow: find
adsorption sites on every surface, generate the H-adsorbed structures,
predict adsorption energies with the built-in TabPFN model, export VASP
POSCAR files and write CSV/JSON summaries.

The demo below loads the manuscript's DFT-derived adsorption energies from
`adsorption_analysis.csv`, so it runs fast and fully offline. To run the
model-driven workflow instead, call
`manager.H_adsorption_analysis(adsorbate="H", output_dir="H_adsorption",
include_visualization=True)` (requires the `ml` extra and a Hugging Face
token for the gated TabPFN model).


In [ ]:
# The manuscript's DFT-derived hydrogen adsorption energies are shipped with
# the package (examples/adsorption_analysis.csv). Loading them keeps the demo
# fast and offline; the full TabPFN prediction workflow is described in the
# README and demonstrated in the companion notebook test_10.4.ipynb.
import pandas as pd

df_ads = pd.read_csv("adsorption_analysis.csv")
print(f"Loaded {len(df_ads)} DFT adsorption records")


In [ ]:
# Facet-resolved adsorption energies and work functions (Mg2Si example).
# The "Eads" column stores the work-function-aligned adsorption free energy
# Gads of the manuscript dataset.
mg2si_sites = df_ads[df_ads["formula"] == "Mg2Si"][
    ["miller_index", "termination", "ads_position", "slab+H", "slab",
     "surface_area", "Eads", "workfunction"]
].sort_values(["miller_index", "ads_position"])
mg2si_sites


### Exchange-current density (i_c0) estimation

`ic0(delta_G, pH, wf)` estimates the exchange current density of the
hydrogen-evolution reaction from a *directly provided* adsorption free
energy `delta_G` (eV), using the BEP-type relation of the manuscript. The
facet work function is used to align the adsorption energy to a common
reference level (3.614 eV).

In [ ]:
from GalvCalc.cathode import ic0, ic0_mg

# Direct estimate from a known/ML-predicted delta_G.
# Example: Mg2Si (110) H6 site, Gads = 0.028 eV, WF = 3.7378 eV.
i0 = ic0(delta_G=0.028, wf=3.7378)
print(f"ic0(delta_G=0.028 eV, wf=3.7378 eV) = {i0:.3e} A/cm^2")

# Backward-compatible wrapper accepting delta_H + correction
i0_legacy, dG = ic0_mg(delta_H=0.028, correction=0.0, wf=3.7378)
print(f"ic0_mg(delta_H=0.028 eV) = {i0_legacy:.3e} A/cm^2, dG = {dG:.3f} eV")


In [ ]:
from GalvCalc.cathode import facet_dependant_property

# Wulff-shape weighted i_c0, offline (pass the bulk Structure instead of a
# Materials Project ID so no API key or network access is needed). Work
# functions and Gads are the DFT values of the manuscript's Mg2Si dataset.
sur_list = [
    {"Formula": "Mg2Si", "Facet": "111_2", "Surface_energy": 0.73, "Work_function": 2.9941},
    {"Formula": "Mg2Si", "Facet": "110_1", "Surface_energy": 0.67, "Work_function": 3.7378},
]
prop_dict = {
    "Mg2Si": {
        "111": {
            "2": {
                "H2": {"Gads": 1.03544},
                "H3": {"Gads": 1.06633},
                "H1": {"Gads": 1.29723},
                "H4": {"Gads": 1.58624},
            }
        },
        "110": {
            "1": {
                "H6": {"Gads": 0.02836},
                "H1": {"Gads": 0.02993},
                "H8": {"Gads": 0.59943},
                "H2": {"Gads": 0.59975},
                "H3": {"Gads": 0.60001},
                "H4": {"Gads": 0.60005},
                "H5": {"Gads": 1.42545},
                "H7": {"Gads": 1.56438},
            }
        },
    }
}

fdp = facet_dependant_property()
weighted_i0 = fdp.weighted_ic(
    sur_list, prop_dict, bulk, correction=0.0, pH=11, ads_criteria="lowest"
)
print(f"Wulff-weighted i_c0 = {weighted_i0:.3e} A/cm^2")
# The (111) termination binds H weakly (large positive Gads) and therefore
# contributes almost no cathodic activity; the (110) termination dominates.


---

## 3. Anode: second-layer substitutional doping

The doping workflow is built on pymatgen's defect-analysis framework:

- `pymatgen.analysis.defects.core.Substitution` performs the substitution.
- `pymatgen.symmetry.analyzer.SpacegroupAnalyzer` selects a
  symmetry-equivalent host site and reports its multiplicity.

By default the dopant is placed in the **second layer** below the surface
(`layer=2`), keeping the outermost surface plane intact.

In [ ]:
from GalvCalc.anode import SurfaceDopingManager

# Create a (001) Mg surface
mg_bulk = Bulk.from_file("Mg.poscar")
mg_surface = Surface.from_bulk(bulk_structure=mg_bulk, miller_index=(0, 0, 1))
manager = SurfaceDopingManager(mg_surface, "Mg_001")

# Batch dope with Zn, Al and Y in the second layer (default)
doped = manager.batch_dope(["Zn", "Al", "Y"], save_to_file=False)

# Inspect the doping records (layer, symmetry multiplicity, depth)
for name, info in manager.doping_info.items():
    print(f"{name}: host {info.host_element} -> {info.dopant_element}, "
          f"site {info.site_index}, layer {info.layer}, "
          f"multiplicity {info.multiplicity}, depth {info.depth:.2f} A")


In [ ]:
# Set DFT-derived descriptors (work function / vacancy energy) and
# compute the electrochemical property table of the doped surfaces.
manager.set_property("Mg_001", "work_function", 3.72)
manager.set_property("Mg_001", "vacancy_energy", 0.84)
manager.set_property("Mg_001_Zn", "work_function", 3.75)
manager.set_property("Mg_001_Zn", "vacancy_energy", 0.79)
manager.set_property("Mg_001_Al", "work_function", 3.74)
manager.set_property("Mg_001_Al", "vacancy_energy", 0.84)
manager.set_property("Mg_001_Y", "work_function", 3.29)
manager.set_property("Mg_001_Y", "vacancy_energy", 0.78)

df_echem = manager.calculate_electrochemical_properties(E00=-2.37, ia00=1e-5)
df_echem[["surface_name", "E0_calculated", "ia0_calculated", "dopant_element"]]


---

## 4. Polarization curves and area-ratio analysis

The polarization module solves the coupled Butler-Volmer kinetics of anodic
dissolution and cathodic hydrogen evolution. Two empirically calibrated
parameterizations are supported:

- `"mg"` (Mg systems): anodic dissolution with n = 2 and exponent
  `alpha_a * n` (no prefactor); the HER cathode uses n = 1 with exponent
  `alpha_c * n` (no prefactor). This is the original manuscript form.
- `"fe"` (Fe systems): both branches use n = 1; the anode exponent is
  `(alpha_a + 1) * n` and both branches carry a global prefactor of 2
  (the calibrated form used for the Fe demo).


In [ ]:
from GalvCalc.polarization import (
    ElectrodeParameters,
    Composition,
    plot_single_polarization,
    plot_comparison_polarization,
    create_mg_based_compositions,
)

# ---- single composition: Fe/Fe2+ + H+/H2 (calibrated Fe form) ----
anode = ElectrodeParameters(4.1e-8, -0.44, 0.5, name="Fe/Fe2+", label=r"Fe/Fe$^{2+}$", kinetic_form="fe")
cath = ElectrodeParameters(7.9e-8, -0.059, 0.5, name="H+/H2", label=r"H$^{+}$/H$_{2}$", kinetic_form="fe")
comp = Composition("Fe", anode=anode, cathodes=[cath], area_ratios=[1, 1])

fig = plot_single_polarization(comp, reference_electrode="SHE")
plt.savefig("demo_output/polarization_single.png", dpi=150, bbox_inches="tight")
print("Saved demo_output/polarization_single.png")
display(Image(filename="demo_output/polarization_single.png"))


In [ ]:
# ---- comparison of Mg-based compositions ----
# The Mg anode uses i0 = 10^-22.2 A/cm^2, alpha_a = 0.55 and n = 2; the pure-Mg
# anode/HER-cathode area ratio is [0.158, 0.842].
comps = create_mg_based_compositions()
fig2 = plot_comparison_polarization(comps, reference_electrode="SCE")
plt.savefig("demo_output/polarization_comparison.png", dpi=150, bbox_inches="tight")
print("Saved demo_output/polarization_comparison.png")
display(Image(filename="demo_output/polarization_comparison.png"))


In [ ]:
# ---- multi-anode / single-cathode plot (Mg kinetics) ----
# Example: 10 alloying elements in an Mg matrix + HER cathode
from GalvCalc.polarization import PolarizationCurvePlotter

mg_anodes = [
    ElectrodeParameters(ei, ep, 0.55, name=nm, kinetic_form="mg")
    for ei, ep, nm in [
        (6e-23, -2.37, "Mg"),
        (1.4e-22, -2.37, "Al"),
        (7.3e-23, -2.359, "Sc"),
        (6.8e-23, -2.35, "Y"),
        (2.7e-22, -2.34, "Zn"),
        (3.1e-23, -2.453, "Gd"),
        (3.3e-23, -2.368, "Nd"),
        (3.4e-23, -2.367, "Mn"),
        (1.5e-22, -2.332, "Zr"),
        (5.8e-23, -2.352, "La"),
    ]
]
her = ElectrodeParameters(10 ** -8.1, -0.61, 0.77, name="H3O+/H2",
                                  label=r"H$_{3}$O$^{+}$/H$_{2}$", kinetic_form="mg")

plotter = PolarizationCurvePlotter(reference_electrode="SHE")
fig3 = plotter.plot_multi_anode_single_cathode(
    composition_name="Mg-X",
    anodes=mg_anodes,
    cathode=her,
    area_ratios=[0.158] * 10 + [0.842],
    potential_range=(-3.5, 0),
    figsize=(10.5, 7.5),
    show_corrosion_point=False,
)
plt.savefig("demo_output/polarization_multi_anode.png", dpi=150, bbox_inches="tight")
print("Saved demo_output/polarization_multi_anode.png")
display(Image(filename="demo_output/polarization_multi_anode.png"))


### Multi-second-phase polarization curves (manuscript style)

The paper's `diff_wt` example compares the polarization curves of nine
Mg alloys, each containing a different second-phase cathode. The anodic
dissolution of the alpha-Mg matrix uses `alpha_a = 0.55` with the
two-electron (n = 2) exponent, the hydrogen-evolution cathode on the
matrix uses `alpha_c = 0.77`, and every phase-specific cathode carries
its own DFT-derived exchange current and transfer coefficient.

In [ ]:
# ---- manuscript-style Mg + second-phase polarization curves ----
# Reproduces the paper's multi-second-phase figure with the manuscript
# kinetic parameters (corrosion points are printed for each alloy).
from GalvCalc.polarization import mg_second_phase_example, plot_mg_second_phases

fig_mg = plot_mg_second_phases(
    mg_second_phase_example(),
    figsize=(9.5, 7.6),
    xlim=(-8.0, 0.0),
    ylim=(-2.8, -1.0),
)
plt.savefig("demo_output/polarization_mg_phases.png", dpi=300, bbox_inches="tight")
print("Saved demo_output/polarization_mg_phases.png")
display(Image(filename="demo_output/polarization_mg_phases.png"))

In [ ]:
# ---- anode / cathode area-ratio optimization ----
from GalvCalc.polarization.area_ratio import (
    AreaRatioAnalyzer,
    create_example_parameters,
)

params = create_example_parameters()
analyzer = AreaRatioAnalyzer(reference_electrode="SCE")
fig4, opt_ratio, max_i = analyzer.plot_area_ratio_analysis(
    params,
    title="Area Ratio Optimization for Mg-based Alloy",
)
print(f"Optimal anode ratio: {opt_ratio:.1%}")
print(f"Max corrosion current: {max_i:.2e} A/cm^2")
plt.savefig("demo_output/area_ratio_optimization.png", dpi=150, bbox_inches="tight")
print("Saved demo_output/area_ratio_optimization.png")
display(Image(filename="demo_output/area_ratio_optimization.png"))


### Alloy-content scan: corrosion vs. alloying-element content

`scan_corrosion_vs_content` evaluates how the corrosion current and potential
change with the weight percent of an alloying element. The volume fraction of
the intermetallic second phase is derived from the solute content
(`wt_to_vol_fraction`), and the Butler-Volmer kinetics are re-fitted at every
content level. The example below reproduces the Mg-Nd / Mg3Nd scan of the
manuscript.

In [ ]:
from GalvCalc.polarization import (
    mg3nd_vol_fraction,
    mg3nd_kinetics,
    scan_corrosion_vs_content,
)

# Nd content sweep (wt%), 0 to 6 %
nd_wt = np.linspace(0, 6, 13)
vol_frac = [mg3nd_vol_fraction(w) for w in nd_wt]

# Dense but still fast scan settings; increase for production use.
df_scan = scan_corrosion_vs_content(
    nd_wt,
    params_fn=lambda w: mg3nd_kinetics(w, mg3nd_vol_fraction(w)),
    domain=(0.0, 1.0),
    n_ratios=51,
    n_potentials=1500,
)
df_scan["vol_fraction"] = vol_frac
df_scan.round(4)


In [ ]:
# Plot the corrosion descriptors vs. Nd content
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.2))

ax1.plot(df_scan["content"], df_scan["max_log10_i_corr"], "o-", color="k", lw=2)
ax1.set_xlabel("Nd content (wt%)")
ax1.set_ylabel(r"log $i_{corr}$ (A/cm$^2$)")

ax2.plot(df_scan["content"], df_scan["E_corr_at_max"], "s-", color="r", lw=2)
ax2.set_xlabel("Nd content (wt%)")
ax2.set_ylabel(r"$E_{corr}$ (V vs. SCE)")

plt.tight_layout()
plt.savefig("demo_output/content_scan.png", dpi=150, bbox_inches="tight")
print("Saved demo_output/content_scan.png")
display(Image(filename="demo_output/content_scan.png"))


---

## 5. ML predictors (optional)

The optional `ml` extra installs `torch`, `scikit-learn` and `tabpfn`, which
power the two pre-trained predictors shipped with the package.

In [ ]:
# CGCNN predictor: surface energies & work functions from a folder of CIF files
# from GalvCalc.predictor import predict_cgcnn
# results = predict_cgcnn(cifpath="surfaces_output", task="regression")
# print(results["predictions"])

# TabPFN predictor: H adsorption energy (eV) for pymatgen structures
# from GalvCalc.predictor import predict
# from pymatgen.core import Structure
# struct = Structure.from_file("H_on_Mg2Si_111_site1.vasp")
# energy = predict(struct)
# print(f"Predicted E_ads = {energy:.2f} eV")
print("See the commented lines above for the ML prediction interfaces.")


---

## Summary

This notebook demonstrated the full GalvCalc workflow:

- **Equilibrium potentials** from a thermodynamic ion database
- **Surface generation, adsorption-site schematics, and 3D viewing**
- **Facet-resolved surface properties and Wulff construction**
- **Hydrogen-adsorption analysis** (DFT dataset) and exchange-current density estimation
- **Second-layer anode surface doping** with electrochemical property tables
- **Polarization curves** (single, comparison, multi-anode) with calibrated
  Mg (n = 2 anode) and Fe (n = 1, prefactor 2) Butler-Volmer forms
- **Area-ratio optimization** and **alloy-content scans** of the corrosion
  current and potential

All figures are saved as PNG files under `demo_output/`. For details of the
underlying models and validation, see the README and the companion notebook
`test_10.4.ipynb`.
